# Análisis TF-IDF y Búsqueda Semántica - Corpus Gutenberg 1000

**Ejercicio 3: Modelo Vectorial y Recuperación de Información**

**Estudiante:** Adrian Correa  
**Fecha:** 29 de abril de 2026

## Objetivos
- Implementar vectorización TF-IDF de 1000 documentos
- Calcular similitud coseno para búsqueda de documentos relevantes
- Analizar distribución de términos y vocabulario
- Evaluar rendimiento del modelo en recuperación de información

## FASE 1: PREPARACIÓN Y CARGA DE DATOS

### Paso 1: Calcular la matriz TF-IDF para corpus de 1000 libros españoles

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os, numpy as np, pandas as pd, string, re, glob, time

stop_words_es = ['el','la','de','que','y','a','en','un','ser','se','no','haber','por','con','su','para','es','al','lo','como','más','o','pero','sus','le','ya','fue','este','ha','porque','esta','son','entre','está','cuando','muy','sin','sobre','tiene','también','me','hasta','hay','donde','han','quien','están','estado','desde','todo','nos','durante','todos','uno','les','ni','contra','otros','fueron','ese','eso','había','ante','ellos','esto','antes']

def limpiar_texto(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans('','',string.punctuation))
    texto = re.sub(r'\d+', '', texto)
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

print("\n" + "="*80)
print("CARGA DE CORPUS: 1000 LIBROS GUTENBERG")
print("="*80)

ruta_corpus = r'D:\7mo semestre\RI\books2\corpus_espanol'
documentos = []
nombres_docs = []

print(f"\nRuta: {ruta_corpus}")
print("Cargando documentos...\n")

tiempo_inicio = time.time()
errors = 0

for archivo in glob.glob(os.path.join(ruta_corpus, '*.txt'))[:1000]:
    try:
        with open(archivo, 'r', encoding='utf-8', errors='ignore') as f:
            texto_limpio = limpiar_texto(f.read()[:25000])
            if len(texto_limpio) > 100:
                documentos.append(texto_limpio)
                nombres_docs.append(os.path.basename(archivo))
        
        if len(documentos) % 200 == 0:
            print(f"  OK: {len(documentos)} documentos")
    except:
        errors += 1

tiempo_carga = time.time() - tiempo_inicio

print(f"\nCARGA COMPLETADA")
print(f"   Documentos: {len(documentos)}")
print(f"   Errores: {errors}")
print(f"   Tiempo: {tiempo_carga:.2f}s")
print(f"   Palabras promedio/doc: {int(np.mean([len(d.split()) for d in documentos]))}")


CARGA DE CORPUS: 1000 LIBROS GUTENBERG

Ruta: D:\7mo semestre\RI\books2\corpus_espanol
Cargando documentos...

  OK: 200 documentos
  OK: 400 documentos
  OK: 600 documentos
  OK: 800 documentos
  OK: 1000 documentos

CARGA COMPLETADA
   Documentos: 1000
   Errores: 0
   Tiempo: 7.28s
   Palabras promedio/doc: 3924


### Paso 2: Cargar corpus Gutenberg 1000 y generar matriz TF-IDF

In [36]:
print("\n" + "="*80)
print("VECTORIZACION TF-IDF")
print("="*80)
print("\nGenerando matriz TF-IDF (esto toma ~10 segundos)...\n")

tiempo_inicio = time.time()

vectorrizador = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_es,
    token_pattern=r'(?u)\b[a-záéíóúñ]{3,}\b',
    min_df=2,
    max_df=0.85,
    ngram_range=(1,2)
)

matriz_tfidf = vectorrizador.fit_transform(documentos)

tiempo_vectorizacion = time.time() - tiempo_inicio

print(f"MATRIZ TF-IDF GENERADA")
print(f"   Dimensiones: {matriz_tfidf.shape[0]} docs x {matriz_tfidf.shape[1]} features")
print(f"   Valores no-cero: {matriz_tfidf.nnz:,}")
print(f"   Densidad: {(matriz_tfidf.nnz / (matriz_tfidf.shape[0] * matriz_tfidf.shape[1])) * 100:.4f}%")
print(f"   Tiempo: {tiempo_vectorizacion:.2f}s")

nombres_features = vectorrizador.get_feature_names_out()
print(f"\nTop 30 terminos mas relevantes:")
print(f"   {list(nombres_features[:30])}")


VECTORIZACION TF-IDF

Generando matriz TF-IDF (esto toma ~10 segundos)...

MATRIZ TF-IDF GENERADA
   Dimensiones: 1000 docs x 5000 features
   Valores no-cero: 689,774
   Densidad: 13.7955%
   Tiempo: 15.93s

Top 30 terminos mas relevantes:
   ['abad', 'abajo', 'abandonado', 'abandonar', 'abandono', 'abierta', 'abierto', 'abismo', 'able', 'abogado', 'about', 'about the', 'above', 'abre', 'abriendo', 'abrigo', 'abril', 'abrir', 'abrió', 'absoluta', 'absolutamente', 'absoluto', 'abuela', 'abuelo', 'abundancia', 'abundante', 'acaba', 'acababa', 'acabado', 'acabar']


### Paso 3: Análisis estadístico y términos más relevantes

In [37]:
print("\n" + "="*80)
print("ANALISIS DE VOCABULARIO")
print("="*80)

tfidf_promedio = np.asarray(matriz_tfidf.mean(axis=0)).ravel()
indices_top = np.argsort(tfidf_promedio)[::-1][:50]

print(f"\nEstadisticas:")
print(f"   Total de features: {len(nombres_features)}")
print(f"   TF-IDF minimo: {matriz_tfidf.min():.6f}")
print(f"   TF-IDF maximo: {matriz_tfidf.max():.6f}")
print(f"   TF-IDF promedio: {tfidf_promedio.mean():.6f}")

print(f"\nTop 25 terminos por relevancia:")
for i, idx in enumerate(indices_top[:25], 1):
    print(f"   {i:2d}. {nombres_features[idx]:20s} -> {tfidf_promedio[idx]:.6f}")


ANALISIS DE VOCABULARIO

Estadisticas:
   Total de features: 5000
   TF-IDF minimo: 0.000000
   TF-IDF maximo: 0.989472
   TF-IDF promedio: 0.003419

Top 25 terminos por relevancia:
    1. the                  -> 0.091904
    2. and                  -> 0.053991
    3. era                  -> 0.052562
    4. qué                  -> 0.042401
    5. tan                  -> 0.039256
    6. usted                -> 0.034235
    7. don                  -> 0.032254
    8. dos                  -> 0.031827
    9. mas                  -> 0.031600
   10. bien                 -> 0.030059
   11. pues                 -> 0.029070
   12. vida                 -> 0.029063
   13. ella                 -> 0.027140
   14. casa                 -> 0.026543
   15. hombre               -> 0.025966
   16. mismo                -> 0.024658
   17. después              -> 0.024209
   18. así                  -> 0.023974
   19. fué                  -> 0.023759
   20. tiempo               -> 0.023255
   21. mis       

### Paso 4: Programar función buscar() - Búsqueda por similitud coseno

In [38]:
print("\n" + "="*80)
print("BUSQUEDA POR SIMILITUD COSENO")
print("="*80)

def buscar_documentos(consulta, k=10):
    consulta_limpia = limpiar_texto(consulta)
    vector_consulta = vectorrizador.transform([consulta_limpia])
    similitudes = cosine_similarity(vector_consulta, matriz_tfidf)[0]
    
    indices_ordenados = np.argsort(similitudes)[::-1][:k]
    resultados = []
    
    for idx in indices_ordenados:
        if similitudes[idx] > 0:
            resultados.append((nombres_docs[idx], similitudes[idx]))
    
    return resultados

consultas_prueba = ['amor pasion', 'misterio secreto', 'aventura viaje', 'muerte tragedia', 'esperanza futuro']

for consulta in consultas_prueba:
    resultados = buscar_documentos(consulta, k=8)
    print(f"\nConsulta: '{consulta}'")
    print(f"   Resultados: {len(resultados)}\n")
    
    if resultados:
        for rank, (nombre_doc, score) in enumerate(resultados, 1):
            barra = '#' * int(score * 40) + '-' * (40 - int(score * 40))
            print(f"   {rank:2d}. [{barra}] {score*100:5.2f}%")
            print(f"       {nombre_doc[:70]}")


BUSQUEDA POR SIMILITUD COSENO

Consulta: 'amor pasion'
   Resultados: 8

    1. [#############---------------------------] 33.52%
       Crónicas de Marianela.txt
    2. [#########-------------------------------] 23.43%
       El arte de amar.txt
    3. [########--------------------------------] 20.95%
       Meditaciones del Quijote.txt
    4. [########--------------------------------] 20.23%
       Místicas; poesías.txt
    5. [#######---------------------------------] 19.62%
       Doctor Sutilis (Cuentos).txt
    6. [#######---------------------------------] 19.60%
       Dulce Dueño.txt
    7. [#######---------------------------------] 18.99%
       El infierno del amor leyenda fantastica.txt
    8. [#######---------------------------------] 18.97%
       Prosas Profanas Obras Completas Vol. II.txt

Consulta: 'misterio secreto'
   Resultados: 8

    1. [###########-----------------------------] 29.96%
       El tesoro misterioso.txt
    2. [##-------------------------------------